In [3]:
# Plantilla de CPU elaborada por Luis Octavio Gonzalez Salcedo, DIplomado IA&AP

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import os
import gzip

def load_mnist(ruta, tipo='train'):


    ruta_categorias = os.path.join(ruta, '%s-labels-idx1-ubyte.gz' % tipo)
    ruta_imagenes = os.path.join(ruta, '%s-images-idx3-ubyte.gz' % tipo)
    
    with gzip.open(ruta_categorias, 'rb') as rut_cat:
        etiquetas = np.frombuffer(rut_cat.read(), dtype=np.uint8, offset=8)

    with gzip.open(ruta_imagenes, 'rb') as rut_imgs:
        imagenes = np.frombuffer(rut_imgs.read(), dtype=np.uint8, offset=16).reshape(len(etiquetas), 784)

    return imagenes, etiquetas

In [ ]:
# Acceso a GoogleDrive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ruta = '/content/drive/MyDrive/Colab Notebooks/fashion_mnist_data'

X_train, Y_train = load_mnist(ruta, tipo='train')
X_test, Y_test = load_mnist(ruta, tipo='test')

In [ ]:
# Reajuste del tamaño de los datos

In [ ]:
X_train=X_train[0:59904,:]
X_test=X_test[0:9984,:]
Y_train=Y_train[0:59904]
Y_test=Y_test[0:9984]

X_train=np.reshape(X_train,(59904,28,28,1))
X_test=np.reshape(X_test,(9984,28,28,1))

In [ ]:
# Importar Tensor FLow 2 (el cual incluye Keras)

In [ ]:
%tensorflow_version 2.x
import tensorflow as tf
print('Versión de tensor flow: '+tf.__version__)

In [ ]:
# Creación del Modelo

In [ ]:
tf.random.set_seed(200)
modelo=tf.keras.models.Sequential()

modelo.add(tf.keras.layers.BatchNormalization(input_shape=X_train.shape[1:]))
modelo.add(tf.keras.layers.Conv2D(64,(5,5), padding='same',activation='elu'))
modelo.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=(2,2)))
modelo.add(tf.keras.layers.Dropout(0.25))

modelo.add(tf.keras.layers.BatchNormalization(input_shape=X_train.shape[1:]))
modelo.add(tf.keras.layers.Conv2D(128,(5,5), padding='same',activation='elu'))
modelo.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=(2,2)))
modelo.add(tf.keras.layers.Dropout(0.25))

modelo.add(tf.keras.layers.BatchNormalization(input_shape=X_train.shape[1:]))
modelo.add(tf.keras.layers.Conv2D(256,(5,5), padding='same',activation='elu'))
modelo.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=(2,2)))
modelo.add(tf.keras.layers.Dropout(0.25))

modelo.add(tf.keras.layers.Flatten())
modelo.add(tf.keras.layers.Dense(256)) #número de neuronas de la red
modelo.add(tf.keras.layers.Activation('elu'))
modelo.add(tf.keras.layers.Dropout(0.5))
modelo.add(tf.keras.layers.Dense(10)) #número de neuronas en la salida
modelo.add(tf.keras.layers.Activation('softmax')) #softmax es para clasificacion de las imagenes

#imprimir en pantalla la info del modelo
modelo.summary()

In [ ]:
# Entrenamiento con CPU

In [ ]:
modelo.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
import timeit
def entrenamiento_cpu():
  with tf.device('/cpu:0'):
    modelo.fit(X_train,Y_train,validation_data=(X_test,Y_test),batch_size=128,epochs=2,verbose=1)
  return None
cpu_time= timeit.timeit('entrenamiento_cpu()', number=1, setup='from __main__ import entrenamiento_cpu')

In [ ]:
print('Tiempo de entrenamiento: ' + str(cpu_time) +'segundos')